# 🐦 Day 02a: HLD — Twitter / News Feed

---

## 🎯 What You'll Master Today
- Tweet storage & timeline generation
- Fan-out on write vs fan-out on read
- Follow graph data model
- Trending topics
- Search architecture

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  TWITTER / NEWS FEED — ARCHITECTURE                              ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Core Problem: User has 500 follows, show their latest tweets    ║
║                                                                   ║
║  Two approaches:                                                 ║
║                                                                   ║
║  Fan-out on WRITE (push):       Fan-out on READ (pull):          ║
║  User tweets → push to          User requests feed →            ║
║  all followers' feeds            fetch from all follows          ║
║  ┌──────┐  write   ┌──────┐    ┌──────┐  read   ┌──────┐     ║
║  │Author│──►each──►│Feed  │    │Reader│──►each──►│Tweet │     ║
║  │tweets│  follower│Cache │    │  req │  follow  │Table │     ║
║  └──────┘          └──────┘    └──────┘          └──────┘     ║
║  ✅ Fast read       ❌ Slow write   ❌ Slow read    ✅ Fast write ║
║  ❌ Celebrity = N writes           ✅ Celebrity OK               ║
║                                                                   ║
║  💡 HYBRID (Twitter's actual approach):                          ║
║  • Regular users: fan-out on write (push to follower caches)    ║
║  • Celebrities (>10K followers): fan-out on read (fetch at read) ║
║  • Merge at read time                                            ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. Scale Estimation
# ============================================================

print("""
📊 TWITTER SCALE ESTIMATION

Assumptions:
  • DAU: 300M users
  • Each user views feed 5 times/day
  • Each user tweets 2 per day (only 20% tweet daily)
  • Average 300 followers per user
  • Tweet size: 140 chars + metadata ≈ 1KB

Read QPS:
  300M × 5 / 86400 ≈ 17,000 reads/sec

Write QPS:
  300M × 0.2 × 2 / 86400 ≈ 1,400 tweets/sec

Fan-out on write:
  1,400 tweets/sec × 300 followers = 420K cache writes/sec (!)
  Celebrity with 10M followers: 1 tweet = 10M writes!

Storage (5 years):
  300M × 0.2 × 2 × 365 × 5 × 1KB ≈ 220 TB

Feed cache:
  300M users × 200 tweet IDs × 8 bytes ≈ 480 GB (fits in Redis cluster)
""")

In [ ]:
# ============================================================
# 2. Fan-out on Write vs Read — Implementation
# ============================================================

from collections import defaultdict
import time

class TwitterFanOutOnWrite:
    """Push model: on tweet, push to every follower's feed cache."""
    
    def __init__(self):
        self.tweets = {}               # tweet_id → tweet
        self.followers = defaultdict(set)  # user → set(followers)
        self.feed_cache = defaultdict(list) # user → [tweet_ids] (pre-built)
        self.tweet_counter = 0
    
    def follow(self, follower, followee):
        self.followers[followee].add(follower)
    
    def post_tweet(self, author, text):
        self.tweet_counter += 1
        tid = self.tweet_counter
        self.tweets[tid] = {"id": tid, "author": author, "text": text, "ts": time.time()}
        
        # Fan-out: push to every follower's cache
        fan_out_count = 0
        for follower in self.followers[author]:
            self.feed_cache[follower].insert(0, tid)  # most recent first
            self.feed_cache[follower] = self.feed_cache[follower][:200]  # keep top 200
            fan_out_count += 1
        
        return tid, fan_out_count
    
    def get_feed(self, user, limit=10):
        """O(1) feed read — already pre-built!"""
        return [self.tweets[tid] for tid in self.feed_cache[user][:limit]]


class TwitterFanOutOnRead:
    """Pull model: on feed request, fetch from all follows."""
    
    def __init__(self):
        self.tweets = {}
        self.following = defaultdict(set)   # user → set(people they follow)
        self.user_tweets = defaultdict(list) # user → [tweet_ids]
        self.tweet_counter = 0
    
    def follow(self, follower, followee):
        self.following[follower].add(followee)
    
    def post_tweet(self, author, text):
        self.tweet_counter += 1
        tid = self.tweet_counter
        self.tweets[tid] = {"id": tid, "author": author, "text": text, "ts": time.time()}
        self.user_tweets[author].insert(0, tid)
        return tid
    
    def get_feed(self, user, limit=10):
        """O(N) — must fetch from all N follows and merge."""
        all_tweets = []
        for followee in self.following[user]:
            for tid in self.user_tweets[followee][:limit]:
                all_tweets.append(self.tweets[tid])
        # Sort by timestamp
        all_tweets.sort(key=lambda t: t['ts'], reverse=True)
        return all_tweets[:limit]

# --- Demo ---
tw = TwitterFanOutOnWrite()
tw.follow("alice", "bob")
tw.follow("alice", "charlie")
tw.follow("dave", "bob")

tid, fan = tw.post_tweet("bob", "Hello World!")
print(f"Bob tweets (id={tid}), pushed to {fan} followers")

tid, fan = tw.post_tweet("charlie", "System Design is fun")
print(f"Charlie tweets (id={tid}), pushed to {fan} followers")

print(f"\nAlice's feed (pre-built, instant):")
for t in tw.get_feed("alice"):
    print(f"  @{t['author']}: {t['text']}")

In [ ]:
# ============================================================
# 3. Data Models
# ============================================================

print("""
📦 DATA MODELS

Users Table (SQL — relational data):
┌──────────────────────────────────────┐
│ user_id  │ username │ email │ bio    │
├──────────────────────────────────────┤
│ BIGINT PK│ VARCHAR  │ ...   │ TEXT   │
└──────────────────────────────────────┘

Tweets Table (SQL or NoSQL):
┌───────────────────────────────────────────────┐
│ tweet_id │ user_id │ text    │ media │ ts     │
├───────────────────────────────────────────────┤
│ BIGINT   │ BIGINT  │ VARCHAR │ URLs  │ TS     │
│ Snowflake│ FK      │ (280)   │ S3    │        │
└───────────────────────────────────────────────┘

Follow Graph (SQL or Graph DB):
┌──────────────────────────────┐
│ follower_id │ followee_id   │
├──────────────────────────────┤
│ BIGINT FK   │ BIGINT FK     │
│ INDEX on both columns       │
└──────────────────────────────┘

Feed Cache (Redis):
  Key: feed:{user_id}
  Value: Sorted Set of tweet_ids (score = timestamp)
  Size: Top 200-800 tweet IDs per user

Media Storage:
  • Images/Videos → S3 (object storage)
  • Served via CDN
  • DB stores only the URL

Trending:
  • Count hashtags in sliding window (last 1hr/24hr)
  • Use Apache Kafka stream → count aggregator
  • Filter by geo-location
""")

In [ ]:
# ============================================================
# 4. Full Architecture Summary
# ============================================================

print("""
╔════════════════════════════════════════════════════════════════╗
║  TWITTER FULL ARCHITECTURE                                   ║
╠════════════════════════════════════════════════════════════════╣
║                                                                ║
║  ┌──────────────────────────────────────────────────┐        ║
║  │                    CDN                           │        ║
║  │              (images, videos)                    │        ║
║  └──────────────────┬───────────────────────────────┘        ║
║                     │                                          ║
║  ┌─────────┐   ┌───┴─────┐   ┌──────────────────┐          ║
║  │ Client  │──►│   LB    │──►│ API Servers       │          ║
║  └─────────┘   └─────────┘   │ (Tweet/Feed/User) │          ║
║                               └──────┬───────────┘          ║
║                        ┌─────────────┼─────────────┐        ║
║                   ┌────▼───┐   ┌─────▼────┐  ┌────▼───┐    ║
║                   │ Redis  │   │   SQL    │  │  S3    │    ║
║                   │ Feed   │   │ Users/   │  │ Media  │    ║
║                   │ Cache  │   │ Tweets   │  │ Store  │    ║
║                   └────────┘   └──────────┘  └────────┘    ║
║                        │                                      ║
║                   ┌────▼────────────────────────────┐        ║
║                   │  Fan-out Service (async)        │        ║
║                   │  Kafka → push to follower caches│        ║
║                   └─────────────────────────────────┘        ║
║                                                                ║
║  Key decisions:                                               ║
║  • Hybrid fan-out (write for reg, read for celebrity)        ║
║  • Snowflake IDs for globally unique tweet IDs               ║
║  • Search: Elasticsearch for full-text search                ║
║  • Notifications: separate service + push queues             ║
║                                                                ║
╚════════════════════════════════════════════════════════════════╝
""")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Design Twitter's news feed | Hybrid fan-out, Redis feed cache, merge at read |
| 2 | Fan-out on write vs read? | Write=fast read, slow write. Read=fast write, slow read. Hybrid=best |
| 3 | Celebrity problem? | 10M followers = 10M writes on tweet. Solution: pull for celebrities |
| 4 | How to handle trending? | Sliding window hashtag counter. Kafka stream processing |
| 5 | Tweet ID generation? | Snowflake (timestamp + machine + sequence). Sortable + unique |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Hybrid fan-out = Twitter's actual approach             │
## │  • Pre-compute feeds in Redis (sorted sets)              │
## │  • Celebrities use pull-based (fan-out on read)          │
## │  • Media → S3 + CDN, DB stores only URLs                │
## │  • Search → Elasticsearch, Trending → stream processing │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **HLD — Instagram**